# Notebook 05 — Embeddings y Transformers

## Objetivo

Comparar modelos más complejos sobre el **subconjunto político**:
- **Parte A:** Embeddings preentrenados (GloVe) + LR/SVM
- **Parte B:** Fine-tuning de DistilBERT

> Los modelos aprenden patrones lingüísticos del dataset; no verifican hechos.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import joblib

from nlp.embeddings import load_glove_vectors, load_or_compute_glove_embeddings
from nlp.io import load_splits
from nlp.metrics import compute_metrics, metrics_to_row
from nlp.plotting import plot_confusion_matrix
from nlp.transformers_data import NewsDataset, prepare_transformer_inputs
from sklearn.metrics import confusion_matrix


## Parte A — Embeddings (GloVe)

In [2]:
EMBEDDING_COLS = ['label', 'title', 'text', 'clean_full_text_without_stopwords']
pol_train, pol_val, pol_test = load_splits('politics', columns=EMBEDDING_COLS)

GLOVE_DIM = 100
TEXT_COL = 'clean_full_text_without_stopwords'


In [3]:
glove = load_glove_vectors(GLOVE_DIM)
print(f'Vectores cargados: {len(glove):,}')


Descargando GloVe (puede tardar varios minutos)...
Extrayendo GloVe...
Parseando GloVe 100d (solo la primera vez)...
GloVe cacheado en glove.6B.100d.kv
Vectores cargados: 400,000


In [4]:
train_texts = pol_train[TEXT_COL].fillna('')
val_texts = pol_val[TEXT_COL].fillna('')
test_texts = pol_test[TEXT_COL].fillna('')

X_train_emb = load_or_compute_glove_embeddings(
    train_texts, DATA_EMBEDDINGS / 'politics_train_glove100d.npz', glove, GLOVE_DIM,
)
X_val_emb = load_or_compute_glove_embeddings(
    val_texts, DATA_EMBEDDINGS / 'politics_val_glove100d.npz', glove, GLOVE_DIM,
)
X_test_emb = load_or_compute_glove_embeddings(
    test_texts, DATA_EMBEDDINGS / 'politics_test_glove100d.npz', glove, GLOVE_DIM,
)
y_train, y_val, y_test = pol_train['label'], pol_val['label'], pol_test['label']


Calculando embeddings -> politics_train_glove100d.npz


Embedding:   0%|          | 0/12635 [00:00<?, ?it/s]

Calculando embeddings -> politics_val_glove100d.npz


Embedding:   0%|          | 0/2707 [00:00<?, ?it/s]

Calculando embeddings -> politics_test_glove100d.npz


Embedding:   0%|          | 0/2708 [00:00<?, ?it/s]

In [5]:

embedding_results = []
for model_name, ModelClass, params in [
    ('logistic_regression', LogisticRegression, {'C': [0.1, 1, 10]}),
    ('linear_svm', LinearSVC, {'C': [0.1, 1, 10]}),
]:
    best_f2, best_clf, best_param = -1, None, None
    for C in params['C']:
        clf = ModelClass(C=C, random_state=RANDOM_STATE, max_iter=2000) if model_name == 'logistic_regression' else ModelClass(C=C, random_state=RANDOM_STATE)
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
        pipe.fit(X_train_emb, y_train)
        y_val_pred = pipe.predict(X_val_emb)
        m = compute_metrics(y_val, y_val_pred)
        if m['f2_fake'] > best_f2:
            best_f2, best_clf, best_param = m['f2_fake'], pipe, C

    y_test_pred = best_clf.predict(X_test_emb)
    y_proba = None
    if hasattr(best_clf.named_steps['clf'], 'predict_proba'):
        y_proba = best_clf.predict_proba(X_test_emb)[:, 1]
    elif hasattr(best_clf.named_steps['clf'], 'decision_function'):
        s = best_clf.decision_function(X_test_emb)
        y_proba = (s - s.min()) / (s.max() - s.min() + 1e-9)

    test_m = compute_metrics(y_test, y_test_pred, y_proba)
    embedding_results.append(metrics_to_row(test_m, {
        'model': model_name, 'representation': 'glove_avg',
        'best_param': best_param, 'dataset_scope': 'politics', 'split': 'test',
    }))

embedding_df = pd.DataFrame(embedding_results)
embedding_df.to_csv(RESULTS_METRICS / 'embedding_results.csv', index=False)
embedding_df


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,representation,best_param,dataset_scope,split
0,0.934638,0.898790,0.905765,0.902264,0.904361,0.974730,logistic_regression,glove_avg,1.0,politics,test
1,0.933900,0.899448,0.902439,0.900941,0.901839,0.974231,linear_svm,glove_avg,0.1,politics,test


## Parte B — Fine-tuning DistilBERT

> **Alcance:** solo subconjunto **politics** (no se entrena sobre el dataset completo).

In [6]:
# Parámetros (config única; ajustar si hay limitaciones computacionales)
SAMPLE_FRAC = 0.1 if DEV_MODE else 1.0
LEARNING_RATES = [2e-5]
BATCH_SIZES = [8]
EPOCHS_LIST = [2]
MAX_LENGTHS = [256]

# Reutilizar checkpoint ya entrenado (evita re-entrenar ~13h en CPU)
REUSE_EXISTING_CHECKPOINT = True
CHECKPOINT_DIR = RESULTS_MODELS / 'distilbert_checkpoints'

if DEV_MODE:
    print('DEV_MODE activo: SAMPLE_FRAC=0.1')
print(f'SAMPLE_FRAC={SAMPLE_FRAC}')
print('Alcance DistilBERT: politics solamente')


DEV_MODE activo: SAMPLE_FRAC=0.1
SAMPLE_FRAC=0.1
Alcance DistilBERT: politics solamente


In [7]:
import json
from pathlib import Path

import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, fbeta_score, precision_recall_fscore_support


def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0,
    )
    f2 = fbeta_score(labels, preds, beta=2, pos_label=1, zero_division=0)
    return {
        'f2_fake': f2,
        'f1_fake': f1[1] if len(f1) > 1 else 0,
        'accuracy': accuracy_score(labels, preds),
    }


In [8]:
# Preparar datos
tr = pol_train.copy()
va = pol_val.copy()
te = pol_test.copy()

if SAMPLE_FRAC < 1.0:
    tr = tr.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    va = va.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    print(f'Muestra reducida: train={len(tr)}, val={len(va)}')

tr_texts = prepare_transformer_inputs(tr)
va_texts = prepare_transformer_inputs(va)
te_texts = prepare_transformer_inputs(te)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Muestra reducida: train=1264, val=271


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def find_best_checkpoint(checkpoint_dir):
    """Busca el mejor checkpoint guardado por Trainer (load_best_model_at_end)."""
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None
    best_path, best_metric = None, -1.0
    for state_file in checkpoint_dir.glob('checkpoint-*/trainer_state.json'):
        state = json.loads(state_file.read_text(encoding='utf-8'))
        metric = state.get('best_metric')
        path = state.get('best_model_checkpoint')
        if metric is not None and metric >= best_metric and path:
            best_metric = metric
            best_path = Path(path)
    return best_path

use_fp16 = torch.cuda.is_available()
transformer_results = []
best_trainer, best_row, best_model = None, None, None
best_f2 = -1

for lr in LEARNING_RATES:
    for bs in BATCH_SIZES:
        for epochs in EPOCHS_LIST:
            for max_len in MAX_LENGTHS:
                print(f'\n=== lr={lr}, bs={bs}, epochs={epochs}, max_len={max_len} ===')
                train_ds = NewsDataset(tr_texts, tr['label'], tokenizer, max_len)
                val_ds = NewsDataset(va_texts, va['label'], tokenizer, max_len)
                test_ds = NewsDataset(te_texts, te['label'], tokenizer, max_len)

                checkpoint_path = find_best_checkpoint(CHECKPOINT_DIR)
                can_reuse = (
                    REUSE_EXISTING_CHECKPOINT
                    and checkpoint_path is not None
                    and checkpoint_path.exists()
                    and lr == 2e-5 and bs == 8 and epochs == 2 and max_len == 256
                )

                if can_reuse:
                    print(f'Reutilizando checkpoint existente: {checkpoint_path}')
                    model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_path))
                    trainer = None
                else:
                    model = AutoModelForSequenceClassification.from_pretrained(
                        'distilbert-base-uncased', num_labels=2
                    )
                    args = TrainingArguments(
                        output_dir=str(CHECKPOINT_DIR),
                        learning_rate=lr,
                        per_device_train_batch_size=bs,
                        per_device_eval_batch_size=bs * 2,
                        num_train_epochs=epochs,
                        eval_strategy='epoch',
                        save_strategy='epoch',
                        load_best_model_at_end=True,
                        metric_for_best_model='f2_fake',
                        greater_is_better=True,
                        logging_steps=50,
                        seed=RANDOM_STATE,
                        report_to='none',
                        dataloader_num_workers=2,
                        fp16=use_fp16,
                    )
                    trainer = Trainer(
                        model=model,
                        args=args,
                        train_dataset=train_ds,
                        eval_dataset=val_ds,
                        data_collator=data_collator,
                        compute_metrics=compute_transformer_metrics,
                        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
                    )
                    trainer.train()
                    model = trainer.model

                pred = Trainer(
                    model=model,
                    data_collator=data_collator,
                ).predict(test_ds)

                y_pred = np.argmax(pred.predictions, axis=-1)
                y_proba = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
                test_m = compute_metrics(te['label'], y_pred, y_proba)
                row = metrics_to_row(test_m, {
                    'model': 'distilbert-base-uncased',
                    'learning_rate': lr, 'batch_size': bs,
                    'epochs': epochs, 'max_length': max_len,
                    'sample_frac': SAMPLE_FRAC,
                    'dataset_scope': 'politics', 'split': 'test',
                    'reused_checkpoint': can_reuse,
                })
                transformer_results.append(row)
                print('Métricas test:', {k: round(v, 4) for k, v in test_m.items() if k != 'roc_auc'})
                if test_m['f2_fake'] > best_f2:
                    best_f2 = test_m['f2_fake']
                    best_trainer = trainer
                    best_row = row
                best_model = model

transformer_df = pd.DataFrame(transformer_results)
transformer_df.to_csv(RESULTS_METRICS / 'transformer_results.csv', index=False)
if best_model is not None:
    best_model.save_pretrained(str(RESULTS_MODELS / 'best_distilbert'))
    tokenizer.save_pretrained(str(RESULTS_MODELS / 'best_distilbert'))
    print('Modelo guardado en results/models/best_distilbert')
transformer_df.sort_values('f2_fake', ascending=False)



=== lr=2e-05, bs=8, epochs=2, max_len=256 ===


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F2 Fake,F1 Fake,Accuracy
1,0.006385,0.002409,1.000000,1.000000,1.000000
2,0.002065,0.001627,1.000000,1.000000,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Métricas test: {'accuracy': 0.9985, 'precision_fake': 0.9989, 'recall_fake': 0.9967, 'f1_fake': 0.9978, 'f2_fake': 0.9971}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en results/models/best_distilbert


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,learning_rate,batch_size,epochs,max_length,sample_frac,dataset_scope,split,reused_checkpoint
0,0.998523,0.998889,0.996674,0.99778,0.997116,0.999986,distilbert-base-uncased,0.00002,8,2,256,0.1,politics,test,False


## Conclusiones

- Los embeddings GloVe capturan semántica pero pueden perder señales de estilo.
- DistilBERT puede capturar contexto más rico; usar `SAMPLE_FRAC < 1.0` si hay limitaciones de hardware.
- Ningún modelo verifica hechos; aprenden patrones del dataset.